# ALTER11 — détection de talents U20, 5 grands championnats

Ce notebook c'est le fil complet de mon analyse : chargement des données brutes,
nettoyage, construction de la base, enrichissement Transfermarkt, ACP, clustering,
et le scoring final (ALTERSCORE). J'ai viré tout ce qui était tâtonnement/tests
ratés — ce qui reste ici, c'est ce qui a vraiment servi.

**Sommaire :**
1. Chargement des données brutes
2. Nettoyage et normalisation
3. Construction de la base SQLite
4. Enrichissement Transfermarkt (position détaillée, valeur marchande)
5. ACP
6. Biplot PC1/PC2
7. PC3 — l'axe que je n'avais pas anticipé au début
8. Clustering K-Means par groupe de postes
9. Coefficient club et scoring ALTERSCORE

**Les limites, pour être honnête direct** : pas de validation prédictive (je n'ai
pas encore vérifié si les scores prédisent quoi que ce soit dans le temps),
le clustering change un peu si je relance avec une autre seed, les pondérations
du score sont choisies à la main pas apprises, et 8 variables c'est peu pour
représenter un joueur. Détail dans le README.


## 0. Setup

In [ ]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Chemins relatifs à la racine du projet (ce notebook est dans notebooks/)
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT_DIR / "data"
DB_PATH = ROOT_DIR / "alter11.db"

print("Librairies chargées ✓")
print(f"Racine du projet : {ROOT_DIR}")


## 1. Chargement des données

Export FBref, 5 grands championnats, saison 2025-2026.


In [ ]:
df_raw = pd.read_csv(
    DATA_DIR / "raw" / "players_data-2025_2026.csv",
    encoding="utf-8",
    sep=","
)

print(f"Shape : {df_raw.shape}")
print(f"Ligues :\n{df_raw['Comp'].value_counts().to_string()}")


## 2. Nettoyage

Rien de compliqué ici : renommage des colonnes FBref, suppression des lignes
parasites (en-têtes répétées, lignes multi-clubs), normalisation nation/position/âge,
et filtre sur les 5 championnats qui m'intéressent.


In [ ]:
rename_map = {
    "Player": "player_name", "Nation": "nation_raw", "Pos": "position_raw",
    "Squad": "team_name", "Comp": "competition", "Age": "age_raw",
    "Born": "birth_year", "MP": "matches_played", "Min": "minutes",
    "90s": "nineties", "Gls": "goals", "Ast": "assists", "Sh": "shots",
    "SoT": "shots_on_target", "Int": "interceptions", "TklW": "tackles_won",
    "CrdY": "yellow_cards", "CrdR": "red_cards", "Fls": "fouls_committed",
    "Fld": "fouls_drawn", "Crs": "crosses", "Mn/MP": "min_per_match",
    "+/-": "plus_minus", "PPM": "points_per_match",
}

cols = {k: v for k, v in rename_map.items() if k in df_raw.columns}
df = df_raw[list(cols.keys())].rename(columns=cols).copy()

print(f"Colonnes gardées : {len(df.columns)}")


In [ ]:
# Supprimer les lignes parasites
df = df[df["player_name"] != "Player"].dropna(subset=["player_name"])
df = df[~df["team_name"].isin(["2TM", "3TM"])].copy()

# Nation : "fr FRA" -> "FRA"
df["nation"] = df["nation_raw"].str.split().str[-1].str.upper()

# Position : "MF,FW" -> "MF"
df["position"] = df["position_raw"].str.split(",").str[0].str.strip()
df["position"] = df["position"].map({"GK": "GK", "DF": "DF", "MF": "MF", "FW": "FW"}).fillna("MF")

# Âge en entier
df["age"] = pd.to_numeric(df["age_raw"], errors="coerce").fillna(0).astype(int)

# Stats en numérique
num_cols = ["birth_year", "matches_played", "minutes", "nineties",
            "goals", "assists", "shots", "shots_on_target",
            "interceptions", "tackles_won", "yellow_cards", "red_cards"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

LIGUES = ["fr Ligue 1", "es La Liga", "it Serie A", "eng Premier League", "de Bundesliga"]
df_l1 = df[df["competition"].isin(LIGUES)].copy()

print(f"Total joueurs 5 ligues : {len(df_l1)}")
print(f"U20 (<=20 ans) : {len(df_l1[df_l1['age'] <= 20])}")


## 3. Base SQLite

Modèle simple : `dim_team`, `dim_player`, `fact_stats`. Rien de sophistiqué,
juste ce qu'il faut pour requêter proprement en SQL plutôt qu'en pandas partout.


In [ ]:
# dim_team
dim_team = df_l1[["team_name", "competition"]].drop_duplicates().reset_index(drop=True)
dim_team.insert(0, "team_id", range(1, len(dim_team) + 1))

# dim_player
df_l1 = df_l1.merge(dim_team[["team_id", "team_name"]], on="team_name", how="left")
dim_player = (
    df_l1[["player_name", "nation", "position", "birth_year", "age", "team_id"]]
    .drop_duplicates(subset=["player_name"])
    .reset_index(drop=True)
)
dim_player.insert(0, "player_id", range(1, len(dim_player) + 1))

# fact_stats
df_l1 = df_l1.merge(dim_player[["player_id", "player_name"]], on="player_name", how="left")
stat_cols = [
    "player_id", "matches_played", "minutes", "nineties",
    "goals", "assists", "shots", "shots_on_target",
    "interceptions", "tackles_won", "yellow_cards", "red_cards",
    "fouls_committed", "fouls_drawn", "min_per_match",
    "plus_minus", "points_per_match", "crosses"
]
fact_stats = df_l1[stat_cols].copy()
fact_stats.insert(0, "stat_id", range(1, len(fact_stats) + 1))

with sqlite3.connect(DB_PATH) as conn:
    dim_team.to_sql("dim_team", conn, if_exists="replace", index=False)
    dim_player.to_sql("dim_player", conn, if_exists="replace", index=False)
    fact_stats.to_sql("fact_stats", conn, if_exists="replace", index=False)

print(f"dim_team   : {len(dim_team)} clubs")
print(f"dim_player : {len(dim_player)} joueurs")
print(f"fact_stats : {len(fact_stats)} lignes")


## 4. Enrichissement Transfermarkt

FBref donne une position générique (DF/MF/FW), pas assez précis pour le clustering
qui suit. Donc scraping Transfermarkt pour avoir la position détaillée (CB, RW, DM...)
et la valeur marchande, avec matching flou (RapidFuzz) parce que les noms ne matchent
jamais parfaitement entre les deux sources.

Je n'exécute pas le scraping ici — ~350 requêtes avec 1.5s de délai entre chacune,
ça prend 10 minutes et ça sollicite Transfermarkt pour rien si le résultat est déjà
en base. Le code est là pour montrer la logique, le vrai run est en commentaire.


In [ ]:
from bs4 import BeautifulSoup
from rapidfuzz import fuzz
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}


def search_transfermarkt(player_name, headers=HEADERS):
    """Cherche un joueur sur Transfermarkt et retourne les candidats trouvés."""
    query = player_name.replace(" ", "+")
    url = f"https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche?query={query}"

    try:
        r = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, "lxml")
        tables = soup.find_all("table", {"class": "items"})
        if not tables:
            return None

        rows = tables[0].find_all("tr", {"class": ["odd", "even"]})
        results = []
        for row in rows:
            cells_ = row.find_all("td")
            if len(cells_) < 9:
                continue
            results.append({
                "tm_name": cells_[2].get_text(strip=True),
                "tm_club": cells_[3].get_text(strip=True),
                "position_detail": cells_[4].get_text(strip=True),
                "age": cells_[6].get_text(strip=True),
                "market_value": cells_[8].get_text(strip=True),
            })
        return results
    except Exception as e:
        print(f"Erreur {player_name}: {e}")
        return None


def best_match(player_name, results, min_score=60):
    """Retourne le meilleur match par similarité de nom (RapidFuzz)."""
    if not results:
        return None, 0
    best, best_score = None, 0
    for r in results:
        score = fuzz.ratio(player_name.lower(), r["tm_name"].lower())
        if score > best_score:
            best, best_score = r, score
    return (best, best_score) if best_score >= min_score else (None, best_score)


# Exécution réelle (désactivée par défaut — voir note ci-dessus) :
# results_list = []
# for _, row in df_u20.iterrows():
#     results = search_transfermarkt(row["player_name"])
#     match, score = best_match(row["player_name"], results)
#     results_list.append({...})
#     time.sleep(1.5)  # politeness delay
#
# df_tm = pd.DataFrame(results_list)
# df_tm.to_csv(ROOT_DIR / "transfermarkt_positions.csv", index=False)

print("Fonctions de scraping définies. Résultat déjà en base (voir alter11.db).")


## 5. ACP

8 variables p90 (offensif/défensif/volume/perf équipe), standardisées, réduites à
3 composantes. Minutes >= 200, gardiens exclus.


In [ ]:
q = """
    SELECT
        p.player_name, p.age, p.position, t.team_name,
        ROUND(f.goals / NULLIF(f.nineties, 0), 2)         AS buts_p90,
        ROUND(f.assists / NULLIF(f.nineties, 0), 2)       AS passes_p90,
        ROUND(f.shots / NULLIF(f.nineties, 0), 2)         AS tirs_p90,
        ROUND(f.tackles_won / NULLIF(f.nineties, 0), 2)   AS tacles_p90,
        ROUND(f.interceptions / NULLIF(f.nineties, 0), 2) AS int_p90,
        ROUND(f.fouls_drawn / NULLIF(f.nineties, 0), 2)   AS fd_p90,
        ROUND(f.minutes * 100.0 / NULLIF(f.matches_played * 90.0, 0), 1) AS min_pct,
        ROUND(f.points_per_match, 2)                      AS ppm
    FROM fact_stats f
    JOIN dim_player p ON f.player_id = p.player_id
    JOIN dim_team t ON p.team_id = t.team_id
    WHERE f.minutes >= 200
      AND p.position != 'GK'
"""

with sqlite3.connect(DB_PATH) as conn:
    df_all = pd.read_sql(q, conn)

print(f"Dataset complet : {df_all.shape[0]} joueurs")
print(f"Dont U20 (<=20 ans) : {(df_all['age'] <= 20).sum()}")


In [ ]:
features = ['buts_p90', 'passes_p90', 'tirs_p90',
            'tacles_p90', 'int_p90', 'fd_p90',
            'min_pct', 'ppm']

df_clean = df_all[features + ['player_name', 'age', 'position']].dropna()

X = df_clean[features].values
noms = df_clean['player_name'].values
ages = df_clean['age'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

print(f"Joueurs après nettoyage : {len(X_pca)}")
print("\nVariance expliquée :")
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1} : {v:.1%}")
print(f"  TOTAL : {sum(pca.explained_variance_ratio_):.1%}")

df_loadings = pd.DataFrame(
    pca.components_.T, index=features, columns=['PC1', 'PC2', 'PC3']
).round(3)
print(f"\nLoadings :\n{df_loadings}")


PC1 (~35%) : offensif vs défensif. PC2 (~16%) : régularité de temps de jeu vs
provoque des fautes. PC3 (~14%) : c'est le plus intéressant — une intensité défensive
indépendante des deux premiers axes. J'y reviens juste après parce que c'est ce qui
fait ressortir les profils box-to-box.


## 6. Biplot PC1/PC2

Chaque point = un joueur, chaque flèche = une variable. La direction de la flèche
dit avec quoi elle corrèle, sa longueur dit son poids. U20 en couleur (dégradé par
âge), seniors en gris pour donner un contexte.


In [ ]:
mask_u20 = ages <= 20

fig, ax = plt.subplots(figsize=(13, 9))

ax.scatter(X_pca[~mask_u20, 0], X_pca[~mask_u20, 1],
           c='lightgrey', alpha=0.3, s=30, zorder=1, label='Seniors')

sc = ax.scatter(X_pca[mask_u20, 0], X_pca[mask_u20, 1],
                c=ages[mask_u20], cmap='RdYlGn_r',
                alpha=0.85, s=80, zorder=3,
                edgecolors='black', linewidths=0.5,
                vmin=16, vmax=20, label='U20')
plt.colorbar(sc, ax=ax, label='Âge')

scale = 3.5
for i, feat in enumerate(features):
    ax.annotate("", xy=(pca.components_[0, i] * scale, pca.components_[1, i] * scale),
                xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='crimson', lw=2))
    ax.text(pca.components_[0, i] * scale * 1.12, pca.components_[1, i] * scale * 1.12,
            feat, fontsize=9, color='crimson', fontweight='bold')

X_pca_u20 = X_pca[mask_u20]
noms_u20 = noms[mask_u20]
for idx in list(np.argsort(X_pca_u20[:, 0])[-6:]) + list(np.argsort(X_pca_u20[:, 0])[:4]):
    ax.annotate(noms_u20[idx], (X_pca_u20[idx, 0], X_pca_u20[idx, 1]),
                fontsize=7.5, fontweight='bold', xytext=(5, 5), textcoords='offset points')

ax.axhline(0, color='grey', lw=0.8, ls='--')
ax.axvline(0, color='grey', lw=0.8, ls='--')
ax.set_xlabel('PC1 — axe offensif/défensif')
ax.set_ylabel('PC2 — axe volume/intensité')
ax.set_title('Biplot ACP — U20 vs Seniors (ALTER11)', fontsize=13, fontweight='bold')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(ROOT_DIR / "biplot_acp_u20.png", dpi=150)
plt.show()


Lamine Yamal, sans surprise, c'est un outlier total (offensif + équipe qui
écrase tout). Plus intéressant pour moi : Robinio Vaz, Kader Meïté, Can Uzun —
profils offensifs réguliers, pas des outliers artificiels. Fofana et Kostoulas
provoquent/prennent beaucoup de fautes. Côté défensif, Liteta, Rodriguez, Scarles
et Louer sortent clairement du lot.


## 7. PC3 — l'axe que je n'avais pas vu venir

Sur PC1/PC2 seuls, les profils box-to-box (actifs offensivement ET défensivement)
se retrouvent noyés dans la moyenne — ils ne ressortent nulle part. PC3 les révèle.
C'est la raison pour laquelle j'ai gardé 3 composantes au lieu de m'arrêter à 2.


In [ ]:
print("=== Loadings PC3 ===")
print(df_loadings[['PC3']].sort_values('PC3', ascending=False).round(3))

fig, ax = plt.subplots(figsize=(13, 9))

ax.scatter(X_pca[~mask_u20, 0], X_pca[~mask_u20, 2],
           c='lightgrey', alpha=0.3, s=30, zorder=1, label='Seniors')
sc = ax.scatter(X_pca[mask_u20, 0], X_pca[mask_u20, 2],
                c=ages[mask_u20], cmap='RdYlGn_r', alpha=0.85, s=80, zorder=3,
                edgecolors='black', linewidths=0.5, vmin=16, vmax=20, label='U20')
plt.colorbar(sc, ax=ax, label='Âge')

scale = 3.5
for i, feat in enumerate(features):
    ax.annotate("", xy=(pca.components_[0, i] * scale, pca.components_[2, i] * scale),
                xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='crimson', lw=2))
    ax.text(pca.components_[0, i] * scale * 1.12, pca.components_[2, i] * scale * 1.12,
            feat, fontsize=9, color='crimson', fontweight='bold')

for idx in list(np.argsort(X_pca_u20[:, 2])[-6:]) + list(np.argsort(X_pca_u20[:, 2])[:4]):
    ax.annotate(noms_u20[idx], (X_pca_u20[idx, 0], X_pca_u20[idx, 2]),
                fontsize=7.5, fontweight='bold', xytext=(5, 5), textcoords='offset points')

ax.axhline(0, color='grey', lw=0.8, ls='--')
ax.axvline(0, color='grey', lw=0.8, ls='--')
ax.set_xlabel('PC1 — axe offensif/défensif')
ax.set_ylabel('PC3 — axe hybride physique')
ax.set_title('Biplot ACP PC1 vs PC3 — U20 vs Seniors (ALTER11)', fontsize=13, fontweight='bold')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(ROOT_DIR / "biplot_acp_pc1_pc3.png", dpi=150)
plt.show()


Bulatović, Fallou Cham, Enzo Koffi Vinette : récupérateurs purs. Le vrai
résultat intéressant ici, c'est **Marc Bernal et Gessime Yassine** — des profils
box-to-box que je n'aurais jamais repérés en restant sur PC1/PC2.


## 8. Clustering par groupe de postes

Je clusterise séparément défenseurs / milieux / attaquants, pas tout le monde
ensemble — sinon le clustering se contente de recréer la distinction de poste au
lieu de trouver des profils de jeu.

k=3 par groupe. Important : j'ai d'abord tourné avec des labels neutres (A/B/C) et
regardé les stats moyennes de chaque cluster **avant** de mettre des noms football
dessus — pas l'inverse. Sinon on invente une histoire avant d'avoir les données.


In [ ]:
q_cluster = """
    SELECT
        p.player_name, p.age, p.position_detail, t.team_name,
        ROUND(f.goals / NULLIF(f.nineties, 0), 2)         AS buts_p90,
        ROUND(f.assists / NULLIF(f.nineties, 0), 2)       AS passes_p90,
        ROUND(f.shots / NULLIF(f.nineties, 0), 2)         AS tirs_p90,
        ROUND(f.tackles_won / NULLIF(f.nineties, 0), 2)   AS tacles_p90,
        ROUND(f.interceptions / NULLIF(f.nineties, 0), 2) AS int_p90,
        ROUND(f.fouls_drawn / NULLIF(f.nineties, 0), 2)   AS fd_p90,
        ROUND(f.minutes * 100.0 / NULLIF(f.matches_played * 90.0, 0), 1) AS min_pct,
        ROUND(f.points_per_match, 2)                      AS ppm,
        p.market_value
    FROM fact_stats f
    JOIN dim_player p ON f.player_id = p.player_id
    JOIN dim_team t ON p.team_id = t.team_id
    WHERE p.age <= 20
      AND f.minutes >= 200
      AND p.position_detail IS NOT NULL
      AND p.position_detail != 'GK'
"""

with sqlite3.connect(DB_PATH) as conn:
    df_cluster = pd.read_sql(q_cluster, conn)

print(f"Dataset clustering : {len(df_cluster)} joueurs")
print(df_cluster['position_detail'].value_counts())


In [ ]:
groupes = {
    'Défenseurs': ['CB', 'LB', 'RB'],
    'Milieux':    ['CM', 'DM', 'AM', 'LM'],
    'Attaquants': ['CF', 'RW', 'LW']
}

# Labels footballistiques assignés après inspection des stats moyennes par cluster
labels_cluster = {
    'Défenseurs': ['Stoppeur', 'Défenseur de ligne', 'Défenseur offensif'],
    'Milieux':    ['Milieu offensif', 'Milieu relayeur', 'Box-to-box'],
    'Attaquants': ['Ailier électrique', 'Renard des surfaces', 'Avant-centre de pointe']
}

df_cluster['cluster_label'] = None
df_cluster['groupe'] = None
scaler_cluster = StandardScaler()

for groupe, postes in groupes.items():
    mask = df_cluster['position_detail'].isin(postes)
    df_g = df_cluster[mask].dropna(subset=features)
    if len(df_g) < 6:
        continue

    X_g = scaler_cluster.fit_transform(df_g[features])
    km = KMeans(n_clusters=3, random_state=42, n_init=10)
    clusters = km.fit_predict(X_g)
    df_cluster.loc[df_g.index, 'groupe'] = groupe

    # Tri des clusters par tirs_p90 croissant -> défensif, neutre, offensif
    cluster_means = {i: df_cluster.loc[df_g.index[clusters == i], 'tirs_p90'].mean()
                      for i in range(3)}
    sorted_clusters = sorted(cluster_means, key=cluster_means.get)

    for rank, cluster_id in enumerate(sorted_clusters):
        idx = df_g.index[clusters == cluster_id]
        df_cluster.loc[idx, 'cluster_label'] = labels_cluster[groupe][rank]

    print(f"\n=== {groupe} ===")
    for label in labels_cluster[groupe]:
        joueurs = df_cluster[df_cluster['cluster_label'] == label]['player_name'].tolist()
        print(f"  {label} ({len(joueurs)}) : {', '.join(joueurs[:8])}{'...' if len(joueurs) > 8 else ''}")

df_cluster.to_csv(ROOT_DIR / "clusters_u20.csv", index=False)
print(f"\nRépartition finale :\n{df_cluster['cluster_label'].value_counts()}")


Projection ACP à 2D par groupe, couleur = cluster.

In [ ]:
couleurs = {
    'Stoppeur': '#2166ac', 'Défenseur de ligne': '#74add1', 'Défenseur offensif': '#abd9e9',
    'Milieu offensif': '#d73027', 'Milieu relayeur': '#fc8d59', 'Box-to-box': '#fee090',
    'Ailier électrique': '#1a9850', 'Renard des surfaces': '#66bd63',
    'Avant-centre de pointe': '#a6d96a',
}

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, groupe in zip(axes, ['Défenseurs', 'Milieux', 'Attaquants']):
    mask = df_cluster['groupe'] == groupe
    df_g = df_cluster[mask].dropna(subset=['cluster_label'])

    X_g = scaler_cluster.fit_transform(df_g[features].fillna(0))
    pca_g = PCA(n_components=2)
    coords = pca_g.fit_transform(X_g)

    for label in df_g['cluster_label'].unique():
        idx = df_g['cluster_label'] == label
        ax.scatter(coords[idx.values, 0], coords[idx.values, 1],
                   c=couleurs.get(label, 'grey'), s=80, alpha=0.8, label=label,
                   edgecolors='black', linewidths=0.4)

    pc1_scores = coords[:, 0]
    for i in list(np.argsort(pc1_scores)[-3:]) + list(np.argsort(pc1_scores)[:2]):
        ax.annotate(df_g.iloc[i]['player_name'], (coords[i, 0], coords[i, 1]),
                    fontsize=6.5, xytext=(4, 4), textcoords='offset points')

    ax.set_title(f'{groupe} ({mask.sum()} joueurs)', fontsize=11, fontweight='bold')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.legend(fontsize=7.5, loc='upper left')
    ax.grid(alpha=0.2)
    ax.axhline(0, color='grey', lw=0.5, ls='--')
    ax.axvline(0, color='grey', lw=0.5, ls='--')

plt.suptitle('Clustering U20 par groupe de postes — ALTER11', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ROOT_DIR / "clustering_u20.png", dpi=150, bbox_inches='tight')
plt.show()


## 9. Coefficient club et scoring final

Un même niveau stat vaut plus dans un club faible que dans une équipe qui écrase
son championnat. Le coefficient club corrige un peu ça à partir du PPM réel de
l'équipe. Plage : 0.70 (gros clubs) à 1.15 (clubs en difficulté) — les bornes sont
choisies à la main, pas optimisées.


In [ ]:
# Points réels 2025-2026 par équipe (5 championnats), saisis manuellement
points_equipes = {
    # Premier League (38 matchs)
    'Arsenal': 85, 'Manchester City': 78, 'Manchester Utd': 71,
    'Aston Villa': 65, 'Liverpool': 60, 'Bournemouth': 57,
    'Sunderland': 54, 'Brighton': 53, 'Brentford': 51, 'Chelsea': 52,
    'Fulham': 49, 'Newcastle United': 48, 'Everton': 46,
    'Leeds United': 44, 'Crystal Palace': 43, 'Nottingham Forest': 42,
    'Tottenham Hotspur': 40, 'West Ham United': 38, 'Burnley': 25, 'Wolves': 22,
    # La Liga (38 matchs)
    'Barcelona': 92, 'Real Madrid': 81, 'Villarreal': 68,
    'Atlético Madrid': 66, 'Real Betis': 62, 'Celta Vigo': 56,
    'Getafe': 52, 'Rayo Vallecano': 50, 'Valencia': 48,
    'Real Sociedad': 47, 'Espanyol': 45, 'Athletic Club': 44,
    'Sevilla': 42, 'Alavés': 40, 'Elche': 38, 'Levante': 36,
    'Osasuna': 34, 'Mallorca': 32, 'Girona': 28, 'Oviedo': 22,
    # Bundesliga (34 matchs)
    'Bayern Munich': 78, 'Leverkusen': 72, 'Dortmund': 62,
    'RB Leipzig': 58, 'Frankfurt': 55, 'Freiburg': 52,
    'Wolfsburg': 48, 'Mainz 05': 46, 'Hoffenheim': 44,
    'Augsburg': 42, 'Werder Bremen': 40, 'Gladbach': 38,
    'Stuttgart': 36, 'Heidenheim': 34, 'Union Berlin': 32,
    'St Pauli': 28, 'Hamburger SV': 24, 'Köln': 20,
    # Ligue 1 (34 matchs)
    'PSG': 86, 'Monaco': 72, 'Marseille': 68, 'Lille': 64, 'Lyon': 60,
    'Nice': 56, 'Lens': 52, 'Rennes': 48, 'Strasbourg': 44, 'Toulouse': 42,
    'Nantes': 40, 'Auxerre': 38, 'Metz': 36, 'Le Havre': 34, 'Brest': 32,
    'Angers': 30, 'Paris FC': 28, 'Lorient': 24,
    # Serie A (38 matchs)
    'Inter': 92, 'Napoli': 84, 'Juventus': 76, 'Milan': 72, 'Lazio': 68,
    'Roma': 64, 'Fiorentina': 60, 'Atalanta': 58, 'Bologna': 54, 'Torino': 50,
    'Udinese': 46, 'Genoa': 44, 'Cagliari': 42, 'Parma': 40, 'Lecce': 38,
    'Hellas Verona': 36, 'Como': 34, 'Sassuolo': 32, 'Pisa': 28, 'Cremonese': 24,
}

matchs_par_ligue = {
    'ENG-Premier League': 38, 'ESP-La Liga': 38, 'FRA-Ligue 1': 34,
    'GER-Bundesliga': 34, 'ITA-Serie A': 38
}

ppm_rows = []
for _, row in dim_team.iterrows():
    if row['team_name'] in points_equipes:
        mp = matchs_par_ligue.get(row['competition'], 38)
        ppm = round(points_equipes[row['team_name']] / mp, 2)
        ppm_rows.append({'team_name': row['team_name'], 'points_per_match': ppm})

df_ppm = pd.DataFrame(ppm_rows)
manquantes = set(dim_team['team_name']) - set(df_ppm['team_name'])
print(f"PPM calculé pour {len(df_ppm)}/{len(dim_team)} équipes")
if manquantes:
    print(f"Équipes sans PPM (à compléter) : {manquantes}")


In [ ]:
ppm_min = df_ppm['points_per_match'].min()
ppm_max = df_ppm['points_per_match'].max()

# Interpolation linéaire entre le pire club (malus 1.15, boost) et le
# meilleur club (malus 0.70, pénalité) — la formule initiale (1 - 0.3*ratio)
# ne descendait jamais sous 1.0 : les clubs faibles n'étaient jamais
# vraiment boostés, juste un peu moins pénalisés. Corrigé pour que la
# plage 0.70-1.15 soit vraiment utilisée dans son intégralité.
df_malus = df_ppm.copy()
df_malus['malus'] = (
    1.15 - 0.45 * (df_malus['points_per_match'] - ppm_min) / (ppm_max - ppm_min)
).round(3)

with sqlite3.connect(DB_PATH) as conn:
    df_malus.to_sql("malus_clubs", conn, if_exists="replace", index=False)

print("Table malus_clubs créée ✓")
print(f"Plage : {df_malus['malus'].min()} (meilleur club) à {df_malus['malus'].max()} (dernier club)")
print(df_malus.sort_values('malus').head(5))


Le scoring lui-même vit dans `sql/03_alterscore.sql` (formule différenciée par
poste, seuils de minutes différents, bonus âge, coef de fiabilité). Je l'appelle ici
via `run_alterscore.py` plutôt que de dupliquer la requête dans le notebook.


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, str(ROOT_DIR / "run_alterscore.py")],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


## Pour la suite

Ça donne une première grille de lecture pour repérer des U20 intéressants. Les
limites (pas de validation prédictive, pondérations à la main, clustering instable
d'un run à l'autre) sont réelles et documentées dans le README — je préfère les
assumer que les cacher.
